In [0]:
#Import Libraries
from pyspark.sql.functions import *

In [0]:
#Read Raw Dataset
df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/workspace/default/uber_data/uber_trips_dataset_50k.csv")

print("Raw Dataset Loaded")

Raw Dataset Loaded


In [0]:
#Data Cleaning
df = df.filter(col("distance_km") > 0)

print("Cleaned Records :", df.count())

Cleaned Records : 49997


In [0]:
#Feature Engineering
df = df.withColumn("pickup_hour", hour("pickup_time"))

df = df.withColumn("pickup_day", dayofweek("pickup_time"))

df = df.withColumn("pickup_month", month("pickup_time"))

In [0]:
#Save Clean Data
df.write.mode("overwrite").format("delta").save(
    "/Volumes/workspace/default/uber_data/etl_output"
)

print("ETL Completed Successfully")

ETL Completed Successfully


In [0]:

#Verify Output
etl_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/uber_data/etl_output"
)

display(etl_df.limit(10))

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time,pickup_hour,pickup_day,pickup_month
1,8270,10683,San Francisco,37.17093115042939,-77.58647936979322,37.1736523397383,-77.61993433646886,2.97,10.71,Completed,Wallet,2023-01-01T00:00:00.000Z,2023-01-01T00:08:54.600Z,0,1,1
2,1860,44743,Boston,38.89812691856606,-108.58297695484846,38.93746379536704,-108.55872717079913,8.43,22.41,Completed,UPI,2023-01-01T00:01:00.000Z,2023-01-01T00:26:17.400Z,0,1,1
3,6390,75839,San Francisco,38.81457056869823,-89.94260270822319,38.82170236125605,-89.8964345566054,5.46,12.91,Completed,Cash,2023-01-01T00:02:00.000Z,2023-01-01T00:18:22.800Z,0,1,1
4,6191,22189,New York,37.29590568598714,-75.32884414927358,37.30137520518609,-75.31748783882725,6.61,15.7,Completed,Wallet,2023-01-01T00:03:00.000Z,2023-01-01T00:22:49.800Z,0,1,1
5,6734,61104,Seattle,38.97239533578873,-121.48291286204801,38.99208841336341,-121.46790441380276,10.5,19.15,Completed,Wallet,2023-01-01T00:04:00.000Z,2023-01-01T00:35:30.000Z,0,1,1
6,7265,84988,San Francisco,37.600550944815616,-106.71906865688572,37.55691580938672,-106.73095912437086,9.94,19.95,Completed,Card,2023-01-01T00:05:00.000Z,2023-01-01T00:34:49.200Z,0,1,1
7,1466,82933,San Francisco,40.28855964243779,-82.8082193858109,40.27883011229549,-82.84463502060092,12.22,25.89,Completed,UPI,2023-01-01T00:06:00.000Z,2023-01-01T00:42:39.600Z,0,1,1
8,5426,60182,Chicago,40.601188277022246,-78.39873323235332,40.569810860072174,-78.36818593976587,10.14,25.55,Completed,Cash,2023-01-01T00:07:00.000Z,2023-01-01T00:37:25.200Z,0,1,1
9,6578,15826,San Francisco,40.29624711069893,-87.57651655291244,40.284637585308246,-87.56478285126917,1.88,7.97,Completed,Cash,2023-01-01T00:08:00.000Z,2023-01-01T00:13:38.400Z,0,1,1
10,9322,63503,Boston,40.71263418106593,-78.49965570372784,40.67026654819631,-78.477475359972,3.7,12.21,No-Show,UPI,2023-01-01T00:09:00.000Z,2023-01-01T00:20:06.000Z,0,1,1


In [0]:
#Save Feature Engineered Dataset
df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/uber_data/feature_engineered_dataset"
)

print("Feature Engineered Dataset Saved")

Feature Engineered Dataset Saved


In [0]:
#Read Feature Engineered Dataset
feature_df = spark.read.parquet(
    "/Volumes/workspace/default/uber_data/feature_engineered_dataset"
)

display(feature_df.limit(5))

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time,pickup_hour,pickup_day,pickup_month
1,8270,10683,San Francisco,37.17093115042939,-77.58647936979322,37.1736523397383,-77.61993433646886,2.97,10.71,Completed,Wallet,2023-01-01T00:00:00.000Z,2023-01-01T00:08:54.600Z,0,1,1
2,1860,44743,Boston,38.89812691856606,-108.58297695484846,38.93746379536704,-108.55872717079913,8.43,22.41,Completed,UPI,2023-01-01T00:01:00.000Z,2023-01-01T00:26:17.400Z,0,1,1
3,6390,75839,San Francisco,38.81457056869823,-89.94260270822319,38.82170236125605,-89.8964345566054,5.46,12.91,Completed,Cash,2023-01-01T00:02:00.000Z,2023-01-01T00:18:22.800Z,0,1,1
4,6191,22189,New York,37.29590568598714,-75.32884414927358,37.30137520518609,-75.31748783882725,6.61,15.7,Completed,Wallet,2023-01-01T00:03:00.000Z,2023-01-01T00:22:49.800Z,0,1,1
5,6734,61104,Seattle,38.97239533578873,-121.48291286204801,38.99208841336341,-121.46790441380276,10.5,19.15,Completed,Wallet,2023-01-01T00:04:00.000Z,2023-01-01T00:35:30.000Z,0,1,1


In [0]:
#Read Prediction Dataset
prediction_df = spark.read.parquet(
    "/Volumes/workspace/default/uber_data/prediction_dataset"
)

display(prediction_df.limit(5))

fare_amount,prediction,prediction_error
2.38,9.797309561516983,7.417309561516983
3.56,13.242487765867065,9.682487765867064
1.61,5.930118625467433,4.320118625467432
3.72,10.06426403004542,6.344264030045419
1.08,4.416699258884817,3.336699258884817
